# Preliminari

In [1]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [2]:
import json
import copy
from pprint import pprint
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from transformers import TextStreamer
from huggingface_hub import login
from google.colab import userdata

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [4]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
#model_name = "unsloth/Llama-3.2-3B-Instruct"
TRAIN_FILE_NAME = "data_luca_train-v1.jsonl"
TEST_FILE_NAME = "data_luca_test-v1.jsonl"
VAL_FILE_NAME = "data_luca_val-v1.jsonl"
PREDICTIONS_FILE_NAME = "data_luca_predictions_2"
FINETUNED_MODEL_NAME = "iltramont/phi-4-v2"
TEMPERATURE = 0.8
MIN_P = 0.1

# Load model

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = FINETUNED_MODEL_NAME, # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

==((====))==  Unsloth 2025.10.1: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/262M [00:00<?, ?B/s]

Unsloth 2025.10.1 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(100352, 5120, padding_idx=100351)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=5120, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=5120, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

# Load dataset

In [6]:
def formatting_messages(messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages,
                                         tokenize=False,
                                         add_generation_prompt=False)


def create_hugging_face_dataset(data: list[dict]) -> Dataset:
    system_content = []
    user_content = []
    assistant_content = []
    formatted_text = []
    for conversation in data:
        formatted_text.append(formatting_messages(conversation['messages']))
        for message in conversation['messages']:
            if message['role'] == 'system':
                system_content.append(message['content'])
            elif message['role'] == 'user':
                user_content.append(message['content'])
            elif message['role'] == 'assistant':
                assistant_content.append(message['content'])
    return Dataset.from_dict(
            {
                'system_content': system_content,
                'user_content': user_content,
                'assistant_content': assistant_content,
                'text': formatted_text
            })

In [7]:
# Carichiamo il nostro file JSON
if "COLAB_" not in "".join(os.environ.keys()):
    test_path = Path('../data/ft-dataset/' + TEST_FILE_NAME)
else:
    test_path = TEST_FILE_NAME

with open(test_path, "r", encoding="utf-8") as f:
    raw_test = [json.loads(line) for line in f]

test_data = copy.deepcopy(raw_test)

for conversation in test_data:
    for message in conversation['messages']:
        if message['role'] == 'assistant':
            message['content'] = json.dumps(message['content'])

# Creiamo ora un "Huggingface dataset" con i nostri dati
dataset_test = create_hugging_face_dataset(test_data)

# Ora abbiamo creato correttamente il nuovo campo "text"
display(dataset_test)
display(dataset_test.features)

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 35
})

{'system_content': Value(dtype='string', id=None),
 'user_content': Value(dtype='string', id=None),
 'assistant_content': Value(dtype='string', id=None),
 'text': Value(dtype='string', id=None)}

# Predictions

In [8]:
if False:
    dataset_test = dataset_test.select(range(3))

In [9]:
dataset_test

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 35
})

## Metodo 1

In [10]:
def tokenize_input(system_content: str,
                   user_content: str,
                   tokenizer=tokenizer):
    messages = [
        {'role': 'system', 'content': system_content},
        {'role': 'user', 'content': user_content}
    ]
    result = tokenizer.apply_chat_template(messages,
                                           tokenize=True,
                                           add_generation_prompt=True,
                                           return_tensors='pt').to('cuda')
    return result

In [11]:
inputs = [tokenize_input(row['system_content'], row['user_content']) for row in dataset_test]

In [12]:
outputs = []
for i, input in enumerate(inputs):
    print(f'Processing input {i+1}')
    inputh_length = input.shape[1]
    output = model.generate(input_ids=input, use_cache=True, temperature=TEMPERATURE, min_p=0.1)
    outputs.append(output[0][inputh_length:-1])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Processing input 1
Processing input 2
Processing input 3
Processing input 4
Processing input 5
Processing input 6
Processing input 7
Processing input 8
Processing input 9
Processing input 10
Processing input 11
Processing input 12
Processing input 13
Processing input 14
Processing input 15
Processing input 16
Processing input 17
Processing input 18
Processing input 19
Processing input 20
Processing input 21
Processing input 22
Processing input 23
Processing input 24
Processing input 25
Processing input 26
Processing input 27
Processing input 28
Processing input 29
Processing input 30
Processing input 31
Processing input 32
Processing input 33
Processing input 34
Processing input 35


In [13]:
tokenizer.decode(outputs[0])

'{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 50.0, "distanza_oai": 60.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "cavallo", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "coinvolgimento_riflessione_peritoneale": "si", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 4.0, "linfonodi_mesorettali": true, "linfonodi_rettali_superiori": false, "linfonodi_mesenterici_inferiori": false, "linfonodi_iliaci_interni": false, "linfonodi_otturatori": false, "linfonodi_sacrali": false, "linfonodi_inguinali_sotto_dentata": false, "linfonodi_inguinali": false, "linfonodi_iliaci_esterni": false, "linfonodi_iliaci_comuni": false, "linfonodi_paraortici": false, "linfonodi_altri": false, "depositi_tumorali": "si", "numero_depositi": 0.0, "emvi_esteso": "si"}'

In [14]:
tokenized_output = [tokenizer.decode(o) for o in outputs]

In [15]:
tokenized_output[34]

'{"morfologia": "solido_anulare", "spessore_parietale": null, "estensione_cranio_caudale": 35.0, "distanza_oai": 30.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "cavallo", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "no", "linfonodi_sospetti": 8.0, "linfonodi_mesorettali": true, "linfonodi_rettali_superiori": false, "linfonodi_mesenterici_inferiori": false, "linfonodi_iliaci_interni": false, "linfonodi_otturatori": true, "linfonodi_sacrali": false, "linfonodi_inguinali_sotto_dentata": false, "linfonodi_inguinali": false, "linfonodi_iliaci_esterni": false, "linfonodi_iliaci_comuni": false, "linfonodi_paraortici": false, "linfonodi_altri": false, "depositi_tumorali": "no", "numero_depositi": 0.0, "emvi_esteso": "no"}'

# Salvataggio risultati

In [16]:
results_dicts = []
for i, o in enumerate(outputs):
    results_dicts.append(
        {
            'model': FINETUNED_MODEL_NAME,
            'temperature': TEMPERATURE,
            'min_p': MIN_P,
            'prediction': tokenizer.decode(o)
        }
    )

In [17]:
type(results_dicts[0]['prediction'])

str

In [18]:
results_dicts[0]['prediction']

'{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 50.0, "distanza_oai": 60.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "cavallo", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "coinvolgimento_riflessione_peritoneale": "si", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 4.0, "linfonodi_mesorettali": true, "linfonodi_rettali_superiori": false, "linfonodi_mesenterici_inferiori": false, "linfonodi_iliaci_interni": false, "linfonodi_otturatori": false, "linfonodi_sacrali": false, "linfonodi_inguinali_sotto_dentata": false, "linfonodi_inguinali": false, "linfonodi_iliaci_esterni": false, "linfonodi_iliaci_comuni": false, "linfonodi_paraortici": false, "linfonodi_altri": false, "depositi_tumorali": "si", "numero_depositi": 0.0, "emvi_esteso": "si"}'

In [19]:
with open(PREDICTIONS_FILE_NAME + '-' + FINETUNED_MODEL_NAME[len('iltramont/'):] + '.jsonl', 'w', encoding='utf-8') as f:
    for result_dict in results_dicts:
        f.write(json.dumps(result_dict) + '\n')